In [1]:
#loading in data
import pandas as pd
import numpy as np

df = pd.read_csv('data/okcupid_profiles.csv')
df = df.dropna(subset=['pets'])
print(df.columns)

Index(['age', 'status', 'sex', 'orientation', 'body_type', 'diet', 'drinks',
       'drugs', 'education', 'ethnicity', 'height', 'income', 'job',
       'last_online', 'location', 'offspring', 'pets', 'religion', 'sign',
       'smokes', 'speaks', 'essay0', 'essay1', 'essay2', 'essay3', 'essay4',
       'essay5', 'essay6', 'essay7', 'essay8', 'essay9'],
      dtype='str')


In [2]:
#getting rid of variables with >10% missing data
missing_counts = df.isna().sum()
missing_pct = df.isna().mean()

df2 = df.loc[:, missing_pct < 0.10]

print(missing_counts)
print(df2.columns)

age                0
status             0
sex                0
orientation        0
body_type       3075
diet           14294
drinks          1016
drugs           9380
education       2950
ethnicity       2985
height             1
income             0
job             3459
last_online        0
location           0
offspring      21707
pets               0
religion       10803
sign            4853
smokes          2506
speaks            21
essay0          3029
essay1          4007
essay2          5159
essay3          6203
essay4          5515
essay5          5642
essay6          7551
essay7          6757
essay8         10797
essay9          6974
dtype: int64
Index(['age', 'status', 'sex', 'orientation', 'body_type', 'drinks',
       'education', 'ethnicity', 'height', 'income', 'job', 'last_online',
       'location', 'pets', 'smokes', 'speaks', 'essay0'],
      dtype='str')


In [3]:
#looking at data structure
cat_cols = df2.drop(['essay0','height','last_online'],axis=1)
print(cat_cols)

for col in cat_cols:
    print(f"\n=== Frequency Table for: {col} ===")
    freq_table = df2[col].value_counts(dropna=False)
    print(freq_table)


       age     status sex orientation       body_type    drinks  \
0       22     single   m    straight  a little extra  socially   
1       35     single   m    straight         average     often   
2       38  available   m    straight            thin  socially   
3       23     single   m    straight            thin  socially   
4       29     single   m    straight        athletic  socially   
...    ...        ...  ..         ...             ...       ...   
59940   31     single   f    straight             NaN  socially   
59941   59     single   f    straight             NaN  socially   
59942   24     single   m    straight             fit     often   
59944   27     single   m    straight        athletic  socially   
59945   39     single   m         gay         average  socially   

                               education            ethnicity  income  \
0          working on college/university         asian, white      -1   
1                  working on space camp         

In [4]:
##########simplifiying complex variables

#ethnicity variable
top5 = df2['ethnicity'].value_counts().nlargest(5).index
print(top5)
df2['ethnicity_simple'] = df2['ethnicity'].apply(
    lambda x: x if pd.isna(x) or x in top5 else 'other'
)

df2['ethnicity_simple'].value_counts(dropna=False)

Index(['white', 'asian', 'hispanic / latin', 'black', 'other'], dtype='str', name='ethnicity')


ethnicity_simple
white               23397
other                7067
asian                3594
NaN                  2985
hispanic / latin     1803
black                1179
Name: count, dtype: int64

In [5]:
# location - city
large_cities = ["new york, new york",
    "los angeles, california",
    "chicago, illinois",
    "houston, texas",
    "phoenix, arizona",
    "philadelphia, pennsylvania",
    "san antonio, texas",
    "san diego, california",
    "dallas, texas",
    "san jose, california",
    "austin, texas",
    "jacksonville, florida",
    "fort worth, texas",
    "columbus, ohio",
    "charlotte, north carolina",
    "san francisco, california",
    "indianapolis, indiana",
    "seattle, washington",
    "denver, colorado",
    "washington, district of columbia"
]

df2['major_city'] = df2['location'].apply(
    lambda x: "yes" if isinstance(x, str) and x.lower() in large_cities
              else (np.nan if pd.isna(x) else "no")
)

df2['major_city'].value_counts(dropna=False)

major_city
yes    20249
no     19776
Name: count, dtype: int64

In [6]:
#location - state
df2['state'] = df2['location'].apply(
    lambda x: x.split(',', 1)[1].strip() if isinstance(x, str) and ',' in x else np.nan
)

df2['state'].value_counts(dropna=False)

state
california              39967
new york                   10
texas                       4
massachusetts               4
oregon                      3
arizona                     3
michigan                    3
illinois                    3
united kingdom              2
georgia                     2
utah                        2
florida                     2
ohio                        2
colorado                    1
hawaii                      1
montana                     1
spain                       1
nevada                      1
vietnam                     1
louisiana                   1
idaho                       1
new jersey                  1
minnesota                   1
washington                  1
west virginia               1
tennessee                   1
rhode island                1
missouri                    1
germany                     1
district of columbia        1
netherlands                 1
Name: count, dtype: int64

In [7]:
#education
def simple_education(x):
    if pd.isna(x):
        return np.nan

    x_lower = x.lower()

    if "college/university" in x_lower:
        return "college/university"
    elif "two-year college" in x_lower:
        return "two-year college"
    elif "masters" in x_lower:
        return "masters"
    elif "high school" in x_lower:
        return "high school"
    elif "law school" in x_lower:
        return "law school"
    elif "ph.d" in x_lower:
        return "ph.d"
    elif "med school" in x_lower:
        return "med school"
    elif "space camp" in x_lower:
        return "space camp"
    else:
        return "other"

df2['education_simple'] = df2['education'].apply(simple_education)

df2['education_simple'].value_counts(dropna=False)

education_simple
college/university    22017
masters                7231
NaN                    2950
two-year college       2239
ph.d                   1524
space camp             1330
high school            1273
law school             1006
med school              455
Name: count, dtype: int64

In [8]:
# pets variable
def simple_pets(x):
    if pd.isna(x):
        return np.nan
    x = x.lower()

    # likes
    has_cat = ("has cat" in x or "likes cat" in x)
    if has_cat:
        return 1
    else:
        return 0


df2['pets_simple'] = df2['pets'].apply(simple_pets)

df2['pets_simple'].value_counts(dropna=False)

pets_simple
1    28623
0    11402
Name: count, dtype: int64

In [9]:
df2.columns

Index(['age', 'status', 'sex', 'orientation', 'body_type', 'drinks',
       'education', 'ethnicity', 'height', 'income', 'job', 'last_online',
       'location', 'pets', 'smokes', 'speaks', 'essay0', 'ethnicity_simple',
       'major_city', 'state', 'education_simple', 'pets_simple'],
      dtype='str')

In [10]:
#saving clean df
df2.to_csv(
        "data/clean_okcupid.csv",
        index=False
    )